# ⚽ Predict the FIFA World Cup 2026

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

Using machine learning, historical statistics, and soccer domain knowledge, predict match scores, corners, and cards for every fixture. You must submit all your predictions before a single ball is kicked.

The scoring system rewards precision: an exact scoreline earns maximum points, while close predictions still earn partial credit. Later rounds carry score multipliers, so a strong model that holds up in the knockout stages can leapfrog the competition. The challenge is designed to be difficult enough that no one can achieve a perfect score—even with AI assistance—but accessible enough that any data enthusiast can participate and score points.

## 💾 The data

You have access to the following files:

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

| Variable | Description |
|---|---|

You may also bring in any external data—FIFA rankings, historical match results, player statistics—to build your predictions.

In [1]:
try:
    from catboost import CatBoostRegressor
except ImportError:
    %pip install catboost
    from catboost import CatBoostRegressor

import os, pickle
import numpy as np
import pandas as pd
import math
import time
from collections import defaultdict
from catboost import CatBoostRegressor
from scipy.special import gammaln
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split

group_fixtures = pd.read_csv('data/group_fixtures.csv')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 15.2 MB/s  0:00:06m0:00:0100:01m

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
knockout_slots = pd.read_csv('data/knockout_slots.csv')
#knockout_slots

## 💪 Competition challenge

The 2026 World Cup has two phases:

- **Group stage** (matches 1–72): The 48 teams are split into 12 groups of 4. Every team plays the other 3 teams in their group once. The best teams from each group advance to the next phase.
- **Knockout stage** (matches 73–104): Single-elimination rounds — Round of 32, Round of 16, Quarter-finals, Semi-finals, and the Final. Lose once and you're out. Crucially, the two teams playing in each knockout match are not known in advance: they depend on who qualified from the group stage.

Submit predictions for **every match** in both phases. For each match you need to predict:

1. **Score** — the exact final scoreline (e.g. `2-1` means the home team scores 2, the away team scores 1). For knockout matches, the score is the result after 90 minutes and extra time — the penalty shootout is not included.
2. **Corners** — the number of corner kicks awarded in the match
3. **Yellow cards** — the number of yellow cards shown in the match
4. **Red cards** — the number of red cards shown in the match

For **group stage** matches, also predict:
- **Winning team** — which team wins the individual match (use `home`, `away`, or `draw`)

For **knockout round** matches, also predict:
- **Matchup** — which two teams you predict will be playing in that slot. Because the bracket is determined by group stage results, you need to predict which teams advance far enough to meet in each round.
- **Match winner** — which team wins the match (use `home` or `away`)
- **Penalties** — whether the match goes to a penalty shootout (`True` or `False`)

### Scoring system

| Category | Condition | Points |
|---|---|---|
| Score | Exact scoreline | 25 |
| Score | Correct goal difference, wrong score | 10 |
| Score | Correct total goals, wrong score | 10 |
| Corners | Exact number | 10 |
| Corners | Off by 2 | 5 |
| Yellow cards | Exact number | 10 |
| Yellow cards | Off by 1 | 5 |
| Red cards | Exact number | 5 |
| Winning team *(group stage only)* | Correct | 40 |
| Matchup *(knockout only)* | Both teams correct | 20 |
| Matchup *(knockout only)* | One team correct | 10 |
| Match winner *(knockout only)* | Correct | 20 |
| Penalties *(knockout only)* | Correct | 5 |

All points for a match are multiplied by the round factor:

| Round | Multiplier |
|---|---|
| Group stage | ×1 |
| Round of 32 | ×1 |
| Round of 16 | ×2 |
| Quarter-final | ×4 |
| Semi-final | ×8 |
| Third-place playoff | ×8 |
| Final | ×16 |

In [3]:
# ─────────────────────────────────────────────────────────────
# FORM FEATURES  (v3: extends history tuple with xG)
#
# History tuple per team: (gf, ga, pts, elo, xgf, xga)
#   [0]=gf  [1]=ga  [2]=pts  [3]=elo  [4]=xgf  [5]=xga
#
# xG fallback: when xg_available=0, rolling xGF/xGA falls back
# to goals scored/conceded — semantically correct for intl data.
# ─────────────────────────────────────────────────────────────

In [4]:
XG_FORM_COLS = [
    'home_form_xgf', 'home_form_xga',
    'away_form_xgf', 'away_form_xga',
    'home_form_xgf_h', 'home_form_xga_h',
    'away_form_xgf_a', 'away_form_xga_a',
]

def add_form_features(df, window=5, seed=None):
    """
    Rolling pre-match features — strictly no leakage.
    History tuple per team: (gf, ga, pts, elo, xgf, xga)

    seed: dict with keys 'all', 'home', 'away', 'last_date' from a prior call.
    """
    import copy
    df = df.sort_values('date').copy()
    df['date'] = pd.to_datetime(df['date'])

    for col in [
        'home_form_goals_for',  'home_form_goals_against',
        'away_form_goals_for',  'away_form_goals_against',
        'home_form_points',     'away_form_points',
        'home_clean_sheet_rate','away_clean_sheet_rate',
        'home_elo_momentum',    'away_elo_momentum',
        'home_form_gf_h', 'home_form_ga_h',
        'away_form_gf_a', 'away_form_ga_a',
        'home_days_rest', 'away_days_rest',
        # v3 additions — rolling xG
        'home_form_xgf', 'home_form_xga',
        'away_form_xgf', 'away_form_xga',
        'home_form_xgf_h', 'home_form_xga_h',
        'away_form_xgf_a', 'away_form_xga_a',
    ]:
        df[col] = 0.0

    # Ensure xG columns exist (backward-compatible with old-format inputs)
    for col in ['home_xg', 'away_xg', 'xg_available']:
        if col not in df.columns:
            df[col] = 0

    if seed:
        all_h  = {t: copy.copy(h[-window:]) for t, h in seed['all'].items()}
        home_h = {t: copy.copy(h[-window:]) for t, h in seed['home'].items()}
        away_h = {t: copy.copy(h[-window:]) for t, h in seed['away'].items()}
        last_d = dict(seed['last_date'])
    else:
        all_h = {}; home_h = {}; away_h = {}; last_d = {}

    DEFAULT_REST = 30

    for idx, row in df.iterrows():
        home, away   = row['home_team'], row['away_team']
        hg, ag       = float(row['home_goals']), float(row['away_goals'])
        h_elo, a_elo = row['home_elo'], row['away_elo']
        match_date   = row['date']

        for t in (home, away):
            all_h.setdefault(t, [])
            home_h.setdefault(t, [])
            away_h.setdefault(t, [])

        h_pts, a_pts = (3, 0) if hg > ag else (0, 3) if hg < ag else (1, 1)

        # xG values — fall back to goals when xG is unavailable
        xg_avail = int(row.get('xg_available', 0))
        h_xgf = float(row['home_xg']) if xg_avail == 1 and pd.notna(row['home_xg']) else hg
        a_xgf = float(row['away_xg']) if xg_avail == 1 and pd.notna(row['away_xg']) else ag

        # ── days rest ────────────────────────────────────────────────
        df.at[idx, 'home_days_rest'] = (
            (match_date - last_d[home]).days if home in last_d else DEFAULT_REST
        )
        df.at[idx, 'away_days_rest'] = (
            (match_date - last_d[away]).days if away in last_d else DEFAULT_REST
        )

        # ── all-venue form ───────────────────────────────────────────
        hh = all_h[home][-window:]
        if hh:
            df.at[idx, 'home_form_goals_for']    = np.mean([m[0] for m in hh])
            df.at[idx, 'home_form_goals_against'] = np.mean([m[1] for m in hh])
            df.at[idx, 'home_form_points']        = np.mean([m[2] for m in hh])
            df.at[idx, 'home_clean_sheet_rate']   = np.mean([m[1] == 0 for m in hh])
            df.at[idx, 'home_elo_momentum']       = h_elo - hh[0][3]
            df.at[idx, 'home_form_xgf']           = np.mean([m[4] for m in hh])
            df.at[idx, 'home_form_xga']           = np.mean([m[5] for m in hh])

        ah = all_h[away][-window:]
        if ah:
            df.at[idx, 'away_form_goals_for']    = np.mean([m[0] for m in ah])
            df.at[idx, 'away_form_goals_against'] = np.mean([m[1] for m in ah])
            df.at[idx, 'away_form_points']        = np.mean([m[2] for m in ah])
            df.at[idx, 'away_clean_sheet_rate']   = np.mean([m[1] == 0 for m in ah])
            df.at[idx, 'away_elo_momentum']       = a_elo - ah[0][3]
            df.at[idx, 'away_form_xgf']           = np.mean([m[4] for m in ah])
            df.at[idx, 'away_form_xga']           = np.mean([m[5] for m in ah])

        # ── venue-split form ─────────────────────────────────────────
        hv = home_h[home][-window:]
        if hv:
            df.at[idx, 'home_form_gf_h']  = np.mean([m[0] for m in hv])
            df.at[idx, 'home_form_ga_h']  = np.mean([m[1] for m in hv])
            df.at[idx, 'home_form_xgf_h'] = np.mean([m[4] for m in hv])
            df.at[idx, 'home_form_xga_h'] = np.mean([m[5] for m in hv])

        av = away_h[away][-window:]
        if av:
            df.at[idx, 'away_form_gf_a']  = np.mean([m[0] for m in av])
            df.at[idx, 'away_form_ga_a']  = np.mean([m[1] for m in av])
            df.at[idx, 'away_form_xgf_a'] = np.mean([m[4] for m in av])
            df.at[idx, 'away_form_xga_a'] = np.mean([m[5] for m in av])

        # ── update histories (6-element tuple) ───────────────────────
        all_h[home].append((hg, ag, h_pts, h_elo, h_xgf, a_xgf))
        all_h[away].append((ag, hg, a_pts, a_elo, a_xgf, h_xgf))
        home_h[home].append((hg, ag, h_pts, h_elo, h_xgf, a_xgf))
        away_h[away].append((ag, hg, a_pts, a_elo, a_xgf, h_xgf))
        last_d[home] = match_date
        last_d[away] = match_date

    history = {'all': all_h, 'home': home_h, 'away': away_h, 'last_date': last_d}
    return df, history


In [5]:
def get_dc_log_lambdas(df, params, team_to_idx):
    n = len(team_to_idx)
    attack   = params[:n]    - np.mean(params[:n])
    defense  = params[n:2*n] - np.mean(params[n:2*n])
    home_adv = params[-2]

    hi = df['home_team'].map(team_to_idx).values
    ai = df['away_team'].map(team_to_idx).values

    log_lh = np.clip(attack[hi] - defense[ai] + home_adv, -10, 10)
    log_la = np.clip(attack[ai] - defense[hi],             -10, 10)
    return log_lh, log_la

In [6]:
# ─────────────────────────────────────────────────────────────
# STAGE 2: CATBOOST POISSON REGRESSION
#
# v3 adds 11 xG-related features (26 → 37 total):
#   - rolling xGF/xGA (all-venue and venue-split)
#   - xG matchup interactions
#   - xg_available flag (teaches CatBoost to weight differently)
# ─────────────────────────────────────────────────────────────

In [7]:
def build_features(df, log_lh, log_la):
    df = df.reset_index(drop=True)

    # Column guards — backward-compatible with old-format DataFrames
    for col in ['home_xg', 'away_xg', 'xg_available']:
        if col not in df.columns:
            df[col] = 0
    for c in XG_FORM_COLS:
        if c not in df.columns:
            df[c] = 0.0
    FIFA_COLS = ['home_avg_attack', 'home_avg_defense', 'away_avg_attack',
                 'away_avg_defense', 'home_avg_overall', 'away_avg_overall']
    for c in FIFA_COLS:
        if c not in df.columns:
            df[c] = np.nan

    return pd.DataFrame({
        # DC base lambdas (Stage 1 signal)
        "log_lambda_home_dc":     log_lh,
        "log_lambda_away_dc":     log_la,
        # ELO
        "elo_diff":               (df["home_elo"] - df["away_elo"]).values,
        "home_elo":               df["home_elo"].values,
        "away_elo":               df["away_elo"].values,
        # All-venue goals form
        "home_form_gf":           df["home_form_goals_for"].values,
        "home_form_ga":           df["home_form_goals_against"].values,
        "away_form_gf":           df["away_form_goals_for"].values,
        "away_form_ga":           df["away_form_goals_against"].values,
        "home_form_pts":          df["home_form_points"].values,
        "away_form_pts":          df["away_form_points"].values,
        "home_cs_rate":           df["home_clean_sheet_rate"].values,
        "away_cs_rate":           df["away_clean_sheet_rate"].values,
        "home_elo_momentum":      df["home_elo_momentum"].values,
        "away_elo_momentum":      df["away_elo_momentum"].values,
        # Venue-split goals form
        "home_form_gf_h":         df["home_form_gf_h"].values,
        "home_form_ga_h":         df["home_form_ga_h"].values,
        "away_form_gf_a":         df["away_form_gf_a"].values,
        "away_form_ga_a":         df["away_form_ga_a"].values,
        # Days rest
        "home_days_rest":         df["home_days_rest"].values,
        "away_days_rest":         df["away_days_rest"].values,
        # Goals matchup interactions
        "home_atk_vs_away_def":   (df["home_form_goals_for"]  - df["away_form_goals_against"]).values,
        "away_atk_vs_home_def":   (df["away_form_goals_for"]  - df["home_form_goals_against"]).values,
        "home_atk_vs_away_def_v": (df["home_form_gf_h"] - df["away_form_ga_a"]).values,
        "away_atk_vs_home_def_v": (df["away_form_gf_a"] - df["home_form_ga_h"]).values,
        # Match context
        "is_intl":                df["is_intl"].values,
        "is_neutral":             df["is_neutral"].values,
        # ── v3: rolling xG form (all-venue) ──────────────────────
        "home_form_xgf":          df["home_form_xgf"].values,
        "home_form_xga":          df["home_form_xga"].values,
        "away_form_xgf":          df["away_form_xgf"].values,
        "away_form_xga":          df["away_form_xga"].values,
        # v3: venue-split xG form
        "home_form_xgf_h":        df["home_form_xgf_h"].values,
        "home_form_xga_h":        df["home_form_xga_h"].values,
        "away_form_xgf_a":        df["away_form_xgf_a"].values,
        "away_form_xga_a":        df["away_form_xga_a"].values,
        # v3: xG matchup interactions
        "home_xgf_vs_away_xga":   (df["home_form_xgf"] - df["away_form_xga"]).values,
        "away_xgf_vs_home_xga":   (df["away_form_xgf"] - df["home_form_xga"]).values,
        # v3: data-source reliability flag
        "xg_available":           df["xg_available"].values,
        # ── v3: FIFA squad ratings (intl only; 0.0 for club via fillna) ─
        "home_avg_attack":        df["home_avg_attack"].fillna(0.0).values,
        "home_avg_defense":       df["home_avg_defense"].fillna(0.0).values,
        "away_avg_attack":        df["away_avg_attack"].fillna(0.0).values,
        "away_avg_defense":       df["away_avg_defense"].fillna(0.0).values,
        "home_avg_overall":       df["home_avg_overall"].fillna(0.0).values,
        "away_avg_overall":       df["away_avg_overall"].fillna(0.0).values,
        "fifa_attack_diff":       (df["home_avg_attack"] - df["away_avg_attack"]).fillna(0.0).values,
        "fifa_defense_diff":      (df["home_avg_defense"] - df["away_avg_defense"]).fillna(0.0).values,
        # Categorical team IDs
        "home_team":              df["home_team"].values,
        "away_team":              df["away_team"].values,
    })

In [8]:
def poisson_prob_matrix(lh, la, max_goals=8):
    lh = max(float(lh), 1e-10)
    la = max(float(la), 1e-10)
    g  = np.arange(max_goals + 1)
    ph = np.exp(-lh + g * np.log(lh) - gammaln(g + 1))
    pa = np.exp(-la + g * np.log(la) - gammaln(g + 1))
    p  = np.outer(ph, pa)
    return p / p.sum()

def outcome_probs(p):
    return np.array([np.tril(p, -1).sum(), np.trace(p), np.triu(p, 1).sum()])

In [9]:
#############################################
# DEDUPLICATION HELPER
#   Used when merging intl_stats + soccerway
#   Prefers soccerway rows (have xG); falls back to intl_stats
#############################################

In [10]:
def dedup_datasets(primary_df, secondary_df):
    """
    Merge two DataFrames, keeping primary rows when duplicates exist.
    Dedup key: (date, home_team, away_team)
    Falls back to (date, home_goals, away_goals) for unresolved name mismatches.
    """
    combined = pd.concat([primary_df, secondary_df], ignore_index=True)
    # Normalize date column — handles mixed str/Timestamp types from CSV reads
    combined['date'] = pd.to_datetime(combined['date'], errors='coerce')
    combined = combined.sort_values(['date', 'home_team', 'away_team'])

    # Exact team-name dedup (keep first = primary / preferred source)
    combined = combined.drop_duplicates(
        subset=['date', 'home_team', 'away_team'], keep='first'
    )
    return combined.sort_values('date').reset_index(drop=True)

In [11]:
qualified_teams = {
    "UEFA Playoff A": "Bosnia and Herzegovina",
    "UEFA Playoff B": "Sweden",
    "UEFA Playoff C": "Turkey",
    "UEFA Playoff D": "Czechia",
    "FIFA Playoff 1": "Congo DR",
    "FIFA Playoff 2": "Iraq",
}

name_map = {
    "USA": "United States",
    "Czechia": "Czech Republic",
    "Congo DR": "Democratic Republic of the Congo",
    "Côte d'Ivoire": "Ivory Coast",
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "United States": "United States",
    "South Korea": "South Korea",
}

# Additional fallback names that are present in this dataset but differ from common FIFA names.
alias_map = {
    "Bosnia-Herzegovina": "Bosnia-Herzegovina",
    "Ivory Coast": "Ivory Coast",
    "Czech Republic": "Czech Republic",
    "South Korea": "South Korea",
    "United States": "United States",
    "Democratic Republic of the Congo": "Democratic Republic of the Congo",
}

def normalize_team_name(name: str) -> str:
    if name in name_map:
        return name_map[name]
    if name in alias_map:
        return alias_map[name]
    return name

In [12]:
INTL_COMPS = [
    "national_team_competition",
    "FIFA World Cup",
    "UEFA Euro",
    "Copa America",
    "African Cup of Nations",
    "Asian Cup",
    "Gold Cup",
]

# Aliases for names that appear differently in match_cards_corners.csv
_CSV_TO_WC: dict[str, str] = {}  # team names already normalised upstream; kept for safety

DECAY_LAMBDA = 0.15   # per-year exponential decay (~0.16× weight for a 2014 match vs 2026)
MIN_OBS      = 3      # minimum rows before trusting a team-specific average


def load_and_filter(path: str, min_year: int = 2014) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["year"] = df["date"].dt.year
    df = df[df["competition"].isin(INTL_COMPS)].copy()
    df = df[df["year"] >= min_year].copy()
    if _CSV_TO_WC:
        df["home_team"] = df["home_team"].replace(_CSV_TO_WC)
        df["away_team"] = df["away_team"].replace(_CSV_TO_WC)
    df["years_ago"] = 2026 - df["year"]
    return df.reset_index(drop=True)


def compute_team_stats(
    df: pd.DataFrame,
    decay_lambda: float = DECAY_LAMBDA,
    min_obs: int = MIN_OBS,
) -> dict:
    all_teams = sorted(set(df["home_team"]) | set(df["away_team"]))
    stats: dict = {}

    def weighted_avg(rows: pd.DataFrame, col: str):
        valid = rows[rows[col].notna()]
        n = len(valid)
        if n < min_obs:
            return None, n
        w = np.exp(-decay_lambda * valid["years_ago"].values)
        return float(np.average(valid[col].values, weights=w)), n

    for team in all_teams:
        hr = df[df["home_team"] == team]
        ar = df[df["away_team"] == team]

        hy, n_hy = weighted_avg(hr, "home_yellow_cards")
        hrd, n_hr = weighted_avg(hr, "home_red_cards")
        hc, n_hc  = weighted_avg(hr, "home_corners")
        ay, n_ay  = weighted_avg(ar, "away_yellow_cards")
        ard, n_ar = weighted_avg(ar, "away_red_cards")
        ac, n_ac  = weighted_avg(ar, "away_corners")

        stats[team] = {
            "home_yellow":  hy,
            "away_yellow":  ay,
            "home_red":     hrd,
            "away_red":     ard,
            "home_corners": hc,
            "away_corners": ac,
        }

    return stats

def global_stats(df: pd.DataFrame) -> dict:
    cards   = df.dropna(subset=["home_yellow_cards"])
    corners = df.dropna(subset=["home_corners"])
    return {
        "home_yellow":  float(cards["home_yellow_cards"].mean()),
        "away_yellow":  float(cards["away_yellow_cards"].mean()),
        "home_red":     float(cards["home_red_cards"].mean()),
        "away_red":     float(cards["away_red_cards"].mean()),
        "home_corners": float(corners["home_corners"].mean()),
        "away_corners": float(corners["away_corners"].mean()),
    }

def predict_cc_match(
    home: str,
    away: str,
    team_stats: dict,
    globals_dict: dict,
) -> dict:
    def get(team: str, side: str, metric: str) -> float:
        val = team_stats.get(team, {}).get(f"{side}_{metric}")
        return val if val is not None else globals_dict[f"{side}_{metric}"]

    hy = get(home, "home", "yellow")
    ay = get(away, "away", "yellow")
    hr = get(home, "home", "red")
    ar = get(away, "away", "red")
    hc = get(home, "home", "corners")
    ac = get(away, "away", "corners")

    return {
        "home_yellow":   int(round(hy)),
        "away_yellow":   int(round(ay)),
        "home_red":      int(round(hr)),
        "away_red":      int(round(ar)),
        "home_corners":  int(round(hc)),
        "away_corners":  int(round(ac)),
        "total_yellow":  int(round(hy + ay)),
        "total_red":     int(round(hr + ar)),
        "total_corners": int(round(hc + ac)),
    }

In [13]:
def build_predictions_for_fixtures(
    fixtures: pd.DataFrame,
    cards_path: str = "match_cards_corners.csv",
    min_year: int = 2014,
) -> tuple[dict, dict, dict]:
    """
    Entry point for predict_wc2026.py.

    Returns:
        preds       — {match_id (int): predict_cc_match(...) result}
        team_stats  — per-team averages (for KO re-use)
        gstats      — global fallback averages
    """
    df     = load_and_filter(cards_path, min_year)
    tstats = compute_team_stats(df)
    gstats = global_stats(df)

    preds: dict = {}
    for _, row in fixtures.iterrows():
        mid  = int(row["match_id"])
        home = str(row["home_team"])
        away = str(row["away_team"])
        preds[mid] = predict_cc_match(home, away, tstats, gstats)

    return preds, tstats, gstats

In [14]:
N_SIMS   = 10_000
RNG_SEED = 42
np.random.seed(RNG_SEED)
# ─────────────────────────────────────────────────────────────
# pretrained model directory
# ─────────────────────────────────────────────────────────────
MODEL_DIR = "models_dc_cat_v3"

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD MODELS
# ─────────────────────────────────────────────────────────────────────────────

def load_model_set(tag):
    pkl_path = f"{MODEL_DIR}/{tag}_dc_params.pkl"
    if not os.path.exists(pkl_path):
        raise FileNotFoundError(
            f"Model not found: {pkl_path}\n"
            f"Run train_final_s3.py first to generate model artifacts."
        )
    with open(pkl_path, "rb") as f:
        saved = pickle.load(f)
    m_home = CatBoostRegressor()
    m_away = CatBoostRegressor()
    m_home.load_model(f"{MODEL_DIR}/{tag}_home.cbm")
    m_away.load_model(f"{MODEL_DIR}/{tag}_away.cbm")
    return m_home, m_away, saved["dc_params"], saved["team_to_idx"]

print("Loading ensemble models ...")
models = {
    "s1": load_model_set("final_s1_intl"),
    "p2": load_model_set("final_p2_sw"),
}
print(f"  S1: {models['s1'][0].tree_count_} home trees, {len(models['s1'][3])} training teams")
print(f"  P2: {models['p2'][0].tree_count_} home trees, {len(models['p2'][3])} training teams")


Loading ensemble models ...
  S1: 271 home trees, 180 training teams
  P2: 1369 home trees, 248 training teams


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. LOAD HISTORICAL DATA & BUILD FORM SEEDS
# ─────────────────────────────────────────────────────────────────────────────

train_intl = pd.read_csv("data/train_intl_v2.csv")
test_intl  = pd.read_csv("data/test_intl_v2.csv")
sw_intl    = pd.read_csv("data/sw_intl.csv")

full_intl = (
    pd.concat([train_intl, test_intl])
    .drop_duplicates(subset=["date", "home_team", "away_team"])
    .sort_values("date").reset_index(drop=True)
)
full_sw = (
    dedup_datasets(sw_intl, full_intl)
    .sort_values("date").reset_index(drop=True)
)

_, hist_s1 = add_form_features(full_intl)
_, hist_p2 = add_form_features(full_sw)

# Last known ELO per team from historical data
last_elo = {}
for _, row in full_intl.sort_values("date").iterrows():
    last_elo[str(row["home_team"])] = float(row["home_elo"])
    last_elo[str(row["away_team"])] = float(row["away_elo"])


In [17]:
fixtures = group_fixtures.copy()
knockout_matches = knockout_slots.copy()

# Resolve playoff placeholders → actual qualified teams
fixtures["home_team"] = (
    fixtures["home_team"].replace(qualified_teams).apply(normalize_team_name)
)
fixtures["away_team"] = (
    fixtures["away_team"].replace(qualified_teams).apply(normalize_team_name)
)

# All 48 WC teams
all_wc_teams = sorted(set(fixtures["home_team"]) | set(fixtures["away_team"]))
team_idx     = {t: i for i, t in enumerate(all_wc_teams)}
n_teams      = len(all_wc_teams)

# Group membership
team_group = {}
group_teams: dict[str, list] = defaultdict(list)
for _, row in fixtures.iterrows():
    for t in (row["home_team"], row["away_team"]):
        if t not in team_group:
            team_group[t] = row["group"]
            group_teams[row["group"]].append(t)

groups = sorted(group_teams.keys())

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. DC LAMBDA HELPER WITH FALLBACK FOR UNSEEN TEAMS
# ─────────────────────────────────────────────────────────────────────────────

def get_dc_lambdas_safe(home, away, dc_params, team_to_idx):
    """DC log-lambdas; unseen teams get mean strength (attack=defense=0)."""
    n         = len(team_to_idx)
    attack    = dc_params[:n]    - np.mean(dc_params[:n])
    defense   = dc_params[n:2*n] - np.mean(dc_params[n:2*n])
    home_adv  = float(dc_params[-2])

    hi = team_to_idx.get(home)
    ai = team_to_idx.get(away)
    atk_h = float(attack[hi])  if hi is not None else 0.0
    def_h = float(defense[hi]) if hi is not None else 0.0
    atk_a = float(attack[ai])  if ai is not None else 0.0
    def_a = float(defense[ai]) if ai is not None else 0.0

    log_lh = float(np.clip(atk_h - def_a + home_adv, -10, 10))
    log_la = float(np.clip(atk_a - def_h,             -10, 10))
    return log_lh, log_la

# ─────────────────────────────────────────────────────────────────────────────
# 5. PREDICT HELPER — single fixture row through S3 ensemble
# ─────────────────────────────────────────────────────────────────────────────

def predict_fixture_row(frow_s1, frow_p2, home, away):
    """
    Given pre-built form-featured rows for S1 and P2 seeds,
    return (lh_ens, la_ens, p_home, p_draw, p_away, predicted_score).
    """
    lh_list, la_list = [], []
    for tag, (m_home, m_away, dc_params, t2i) in models.items():
        frow = frow_s1 if tag == "s1" else frow_p2
        log_lh, log_la = get_dc_lambdas_safe(home, away, dc_params, t2i)
        X   = build_features(frow.reset_index(drop=True),
                             np.array([log_lh]), np.array([log_la]))
        lh  = float(np.clip(m_home.predict(X), 0.05, 10)[0])
        la  = float(np.clip(m_away.predict(X), 0.05, 10)[0])
        lh_list.append(lh)
        la_list.append(la)

    lh_ens = float(np.mean(lh_list))
    la_ens = float(np.mean(la_list))
    pm     = poisson_prob_matrix(lh_ens, la_ens)
    probs  = outcome_probs(pm)
    ph, pa = np.unravel_index(np.argmax(pm), pm.shape)
    return lh_ens, la_ens, float(probs[0]), float(probs[1]), float(probs[2]), f"{ph}-{pa}"

# ─────────────────────────────────────────────────────────────────────────────
# 6. PRECOMPUTE GROUP STAGE PREDICTIONS (frozen form at training cutoff)
# ─────────────────────────────────────────────────────────────────────────────

def build_fixture_frame(home, away, date, hist, is_neutral=0):
    """Build a 1-row DataFrame suitable for add_form_features."""
    row = pd.DataFrame({
        "date":       [date],
        "home_team":  [home],
        "away_team":  [away],
        "home_goals": [0.0],
        "away_goals": [0.0],
        "home_elo":   [last_elo.get(home, 1500.0)],
        "away_elo":   [last_elo.get(away, 1500.0)],
        "home_xg":    [0.0],
        "away_xg":    [0.0],
        "xg_available": [0],
        "is_intl":    [1],
        "is_neutral": [is_neutral],
    })
    result, _ = add_form_features(row, seed=hist)
    return result

group_preds = {}  # match_id → dict with lh, la, probs, score

for _, fx in fixtures.iterrows():
    mid  = int(fx["match_id"])
    home = fx["home_team"]
    away = fx["away_team"]
    date = str(fx["date_utc"])[:10]

    frow_s1 = build_fixture_frame(home, away, date, hist_s1)
    frow_p2 = build_fixture_frame(home, away, date, hist_p2)
    lh, la, p_h, p_d, p_a, score = predict_fixture_row(frow_s1, frow_p2, home, away)

    group_preds[mid] = {
        "match_id": mid, "group": fx["group"],
        "home_team": home, "away_team": away,
        "lh": lh, "la": la,
        "p_home": p_h, "p_draw": p_d, "p_away": p_a,
        "predicted_score": score,
    }



In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# 6b. CARDS & CORNERS — GROUP STAGE
# ─────────────────────────────────────────────────────────────────────────────

_cc_preds, _cc_team_stats, _cc_globals = build_predictions_for_fixtures(
    fixtures, cards_path="data/match_cards_corners.csv", min_year=2014
)
for mid, cc in _cc_preds.items():
    group_preds[mid].update(cc)

_cc_df = pd.DataFrame(list(_cc_preds.values()))

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. PRECOMPUTE KNOCKOUT MATCH LAMBDAS (all 48×48 pairs, neutral form)
# ─────────────────────────────────────────────────────────────────────────────

KNOCKOUT_DATE = "2026-07-05"

ordered_pairs = [(h, a) for h in all_wc_teams for a in all_wc_teams if h != a]

# Build batch DataFrame for all pairs with neutral form (history-free)
batch_rows = [{
    "date": KNOCKOUT_DATE, "home_team": h, "away_team": a,
    "home_goals": 0.0, "away_goals": 0.0,
    "home_elo": last_elo.get(h, 1500.0), "away_elo": last_elo.get(a, 1500.0),
    "home_xg": 0.0, "away_xg": 0.0,
    "xg_available": 0, "is_intl": 1, "is_neutral": 1,
} for h, a in ordered_pairs]

batch_df, _ = add_form_features(pd.DataFrame(batch_rows))

# Accumulate lambdas from both models (each contributes 50%)
ko_lh = np.zeros(len(ordered_pairs))
ko_la = np.zeros(len(ordered_pairs))

for tag, (m_home, m_away, dc_params, t2i) in models.items():
    log_lh_all = np.array([get_dc_lambdas_safe(h, a, dc_params, t2i)[0] for h, a in ordered_pairs])
    log_la_all = np.array([get_dc_lambdas_safe(h, a, dc_params, t2i)[1] for h, a in ordered_pairs])
    X_batch    = build_features(batch_df.reset_index(drop=True), log_lh_all, log_la_all)
    ko_lh     += np.clip(m_home.predict(X_batch), 0.05, 10) / 2
    ko_la     += np.clip(m_away.predict(X_batch), 0.05, 10) / 2

# Build lookup matrices [home_team_idx, away_team_idx]
ko_lh_mat = np.full((n_teams, n_teams), 1.2)
ko_la_mat = np.full((n_teams, n_teams), 0.9)
for i, (h, a) in enumerate(ordered_pairs):
    hi, ai = team_idx[h], team_idx[a]
    ko_lh_mat[hi, ai] = ko_lh[i]
    ko_la_mat[hi, ai] = ko_la[i]

# ─────────────────────────────────────────────────────────────────────────────
# 8. MONTE CARLO SIMULATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

# Pre-index group stage for fast vectorised sampling
match_ids_ordered = list(fixtures["match_id"].astype(int))
lh_vec = np.array([group_preds[m]["lh"] for m in match_ids_ordered])
la_vec = np.array([group_preds[m]["la"] for m in match_ids_ordered])

# Map match → group, home, away for standings accumulation
match_meta = {
    m: (group_preds[m]["group"],
        group_preds[m]["home_team"],
        group_preds[m]["away_team"])
    for m in match_ids_ordered
}

# Group→list of match_ids
group_match_ids: dict[str, list] = defaultdict(list)
for mid in match_ids_ordered:
    group_match_ids[group_preds[mid]["group"]].append(mid)


def rank_group(pts_dict, gd_dict, gf_dict):
    """Sort 4 teams: pts desc → GD desc → GF desc → alphabetical."""
    teams = list(pts_dict.keys())
    teams.sort(key=lambda t: (-pts_dict[t], -gd_dict[t], -gf_dict[t], t))
    return teams  # [1st, 2nd, 3rd, 4th]


def simulate_group_stage(hg_all, ag_all):
    """
    Given sampled goals for all 72 matches, return:
      group_rankings: {group: [1st, 2nd, 3rd, 4th]}
    """
    pts = {g: defaultdict(int) for g in groups}
    gd  = {g: defaultdict(int) for g in groups}
    gf  = {g: defaultdict(int) for g in groups}

    for i, mid in enumerate(match_ids_ordered):
        grp, home, away = match_meta[mid]
        hg, ag = int(hg_all[i]), int(ag_all[i])

        if hg > ag:
            pts[grp][home] += 3
        elif ag > hg:
            pts[grp][away] += 3
        else:
            pts[grp][home] += 1
            pts[grp][away] += 1

        gd[grp][home] += hg - ag;  gd[grp][away] += ag - hg
        gf[grp][home] += hg;       gf[grp][away] += ag

    # Ensure all 4 group members appear (default 0 for groups with no goals yet)
    for g, team_list in group_teams.items():
        for t in team_list:
            pts[g].setdefault(t, 0)
            gd[g].setdefault(t, 0)
            gf[g].setdefault(t, 0)

    group_rankings = {g: rank_group(pts[g], gd[g], gf[g]) for g in groups}
    return group_rankings, pts, gd, gf


def get_third_qualifiers(group_rankings, raw_pts, raw_gd, raw_gf):
    """
    Best 8 of 12 third-place teams, ranked by pts → GD → GF → group letter.
    Returns list of 8 names sorted by group letter for consistent slot assignment.
    """
    thirds = []
    for g, ranked in group_rankings.items():
        if len(ranked) >= 3:
            third = ranked[2]
            thirds.append({
                "team": third, "group": g,
                "pts": raw_pts[g][third],
                "gd":  raw_gd[g][third],
                "gf":  raw_gf[g][third],
            })
    # Rank cross-group by performance, then take best 8
    thirds.sort(key=lambda x: (-x["pts"], -x["gd"], -x["gf"], x["group"]))
    best8 = thirds[:8]
    # Sort qualified 8 by group letter for deterministic slot assignment
    best8.sort(key=lambda x: x["group"])
    return [t["team"] for t in best8]


def fill_r32_bracket(group_rankings, third_qualifiers):
    """
    Map group results + 3rd-place qualifiers to R32 slot match pairs.
    third_qualifiers: list of 8 team names sorted by group letter.
    Returns {match_id: (home_team, away_team)}.
    """
    slot_map = {}
    for g, ranked in group_rankings.items():
        if len(ranked) >= 1: slot_map[f"Winner Group {g}"]  = ranked[0]
        if len(ranked) >= 2: slot_map[f"Runner-up Group {g}"] = ranked[1]

    r32_rows = (
        knockout_slots[knockout_slots["round"] == "Round of 32"]
        .sort_values("match_id")
    )
    bracket = {}
    third_ptr = 0

    def resolve(slot):
        nonlocal third_ptr
        if slot in slot_map:
            return slot_map[slot]
        if "Best 3rd" in slot:
            team = third_qualifiers[third_ptr] if third_ptr < len(third_qualifiers) else None
            third_ptr += 1
            return team
        return None

    for _, row in r32_rows.iterrows():
        mid  = int(row["match_id"])
        home = resolve(row["slot_home"])
        away = resolve(row["slot_away"])
        if home and away:
            bracket[mid] = (home, away)

    return bracket


def simulate_knockout_match(home, away):
    """
    Sample Poisson goals for one knockout match.
    If tied after 90 min → 50/50 penalty.
    Returns (winner, home_goals, away_goals, on_pens: bool).
    """
    hi, ai = team_idx.get(home, 0), team_idx.get(away, 0)
    # Average both bracket orientations so the result is truly neutral
    lh = (ko_lh_mat[hi, ai] + ko_la_mat[ai, hi]) / 2
    la = (ko_la_mat[hi, ai] + ko_lh_mat[ai, hi]) / 2
    #lh = ko_lh_mat[hi, ai]
    #la = ko_la_mat[hi, ai]
    hg = np.random.poisson(lh)
    ag = np.random.poisson(la)
    if hg > ag:
        return home, hg, ag, False
    elif ag > hg:
        return away, hg, ag, False
    else:
        winner = home if np.random.random() < 0.5 else away
        return winner, hg, ag, True


def run_one_simulation():
    """
    Full tournament simulation.
    Returns:
      match_winner   {match_id: team}       — all 32 knockout match winners
      match_loser    {match_id: team}       — all 32 knockout match losers
      match_score    {match_id: "H-A"}
      match_pens     {match_id: bool}       — True if went to penalties
      group_rankings {group: [1st,..,4th]}
    """
    # ── Group stage ──────────────────────────────────────────────────────────
    hg_all = np.random.poisson(lh_vec)
    ag_all = np.random.poisson(la_vec)
    group_rankings, raw_pts, raw_gd, raw_gf = simulate_group_stage(hg_all, ag_all)

    third_qualifiers = get_third_qualifiers(group_rankings, raw_pts, raw_gd, raw_gf)
    r32_bracket      = fill_r32_bracket(group_rankings, third_qualifiers)

    # ── Knockout rounds ───────────────────────────────────────────────────────
    current_matches = dict(r32_bracket)  # match_id → (home, away)
    match_winner     = {}
    match_loser      = {}
    match_score      = {}
    match_pens       = {}
    match_home_teams = {}  # track which team was designated home in each KO slot

    round_groups = {
        "Round of 32": list(
            knockout_matches[knockout_matches["round"] == "Round of 32"]["match_id"].astype(int)
        ),
        "Round of 16": list(
            knockout_matches[knockout_matches["round"] == "Round of 16"]["match_id"].astype(int)
        ),
        "Quarter-final": list(
            knockout_matches[knockout_matches["round"] == "Quarter-final"]["match_id"].astype(int)
        ),
        "Semi-final": list(
            knockout_matches[knockout_matches["round"] == "Semi-final"]["match_id"].astype(int)
        ),
        "Third-place playoff": [103],
        "Final": [104],
    }

    for round_name, round_mids in round_groups.items():
        for mid in round_mids:
            if mid not in current_matches:
                continue
            home, away       = current_matches[mid]
            w, hg, ag, pens  = simulate_knockout_match(home, away)
            loser            = away if w == home else home
            match_winner[mid]     = w
            match_loser[mid]      = loser
            match_score[mid]      = f"{hg}-{ag}" + (" (pens)" if pens else "")
            match_pens[mid]       = pens
            match_home_teams[mid] = home

        # Propagate winners/losers to next round slots
        next_rows = knockout_matches[
            knockout_matches["round"].isin(["Round of 16", "Quarter-final",
                                          "Semi-final", "Final",
                                          "Third-place playoff"])
        ]
        for _, row in next_rows.iterrows():
            mid = int(row["match_id"])
            def parse_team(slot):
                if slot.startswith("Winner Match "):
                    ref = int(slot.split()[-1])
                    return match_winner.get(ref)
                if slot.startswith("Loser Match "):
                    ref = int(slot.split()[-1])
                    return match_loser.get(ref)
                return None
            h = parse_team(row["slot_home"])
            a = parse_team(row["slot_away"])
            if h and a:
                current_matches[mid] = (h, a)

    return match_winner, match_loser, match_score, match_pens, group_rankings, match_home_teams


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. RUN MONTE CARLO
# ─────────────────────────────────────────────────────────────────────────────

t0 = time.time()

ROUNDS = ["Group Stage", "Round of 32", "Round of 16", "Quarter-final",
          "Semi-final", "Final", "Winner"]
advance_counts = {r: defaultdict(int) for r in ROUNDS}
ko_bracket_wins = defaultdict(lambda: defaultdict(int))  # match_id → team → count
group_rank_counts = defaultdict(lambda: defaultdict(int))  # team → "1st"/"2nd"/"3rd"/"4th" → count

ko_pens_counts = defaultdict(int)
ko_home_counts = defaultdict(lambda: defaultdict(int))
ko_away_counts = defaultdict(lambda: defaultdict(int))

all_ko_mid = list(knockout_matches["match_id"].astype(int))
r32_mids   = list(knockout_matches[knockout_matches["round"] == "Round of 32"]["match_id"].astype(int))
r16_mids   = list(knockout_matches[knockout_matches["round"] == "Round of 16"]["match_id"].astype(int))
qf_mids    = list(knockout_matches[knockout_matches["round"] == "Quarter-final"]["match_id"].astype(int))
sf_mids    = list(knockout_matches[knockout_matches["round"] == "Semi-final"]["match_id"].astype(int))

for sim in range(N_SIMS):
    win, lose, score, pens, g_rank, home_teams = run_one_simulation()

    # Track group stage qualifiers
    for g, ranked in g_rank.items():
        for pos, team in enumerate(ranked, 1):
            advance_counts["Group Stage"][team] += 1
            group_rank_counts[team][str(pos)] += 1

    # Track knockout advancement

    for mid in r32_mids:
        if mid in win: advance_counts["Round of 32"][win[mid]] += 1
    for mid in r16_mids:
        if mid in win: advance_counts["Round of 16"][win[mid]] += 1
    for mid in qf_mids:
        if mid in win: advance_counts["Quarter-final"][win[mid]] += 1
    for mid in sf_mids:
        if mid in win: advance_counts["Semi-final"][win[mid]] += 1
    if 104 in win:
        advance_counts["Final"][win[104]] += 1
        advance_counts["Winner"][win[104]] += 1

    # Track most common bracket path
    for mid in all_ko_mid:
        if mid in win:
            ko_bracket_wins[mid][win[mid]] += 1

    # Track penalty frequency and home/away identity per KO slot
    for mid in all_ko_mid:
        if pens.get(mid, False):
            ko_pens_counts[mid] += 1
        if mid in home_teams and mid in win:
            h = home_teams[mid]
            a = lose[mid] if win[mid] == h else win[mid]
            ko_home_counts[mid][h] += 1
            ko_away_counts[mid][a] += 1

    if (sim + 1) % 2000 == 0:
        elapsed = time.time() - t0
        print(f"  {sim+1:,}/{N_SIMS:,}  ({elapsed:.0f}s)")

elapsed = time.time() - t0

  2,000/10,000  (18s)
  4,000/10,000  (36s)
  6,000/10,000  (54s)
  8,000/10,000  (72s)
  10,000/10,000  (91s)


In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. OUTPUT: GROUP PREDICTIONS CSV
# ─────────────────────────────────────────────────────────────────────────────

group_pred_rows = []
for mid in match_ids_ordered:
    p = group_preds[mid]

    _h, _a = map(int, p["predicted_score"].split("-"))
    if _h > _a:
        pred_winner = "home"
    elif _a > _h:
        pred_winner = "away"
    else:
        pred_winner = "draw"

    group_pred_rows.append({
        "match_id":            mid,
        "group":               p["group"],
        "home_team":           p["home_team"],
        "away_team":           p["away_team"],
        "expected_home_goals": round(p["lh"], 3),
        "expected_away_goals": round(p["la"], 3),
        "predicted_score":     p["predicted_score"],
        "prob_home_win":       round(p["p_home"], 4),
        "prob_draw":           round(p["p_draw"], 4),
        "prob_away_win":       round(p["p_away"], 4),
        "predicted_winner":         pred_winner,
        "predicted_home_yellow":    p.get("home_yellow",  0),
        "predicted_away_yellow":    p.get("away_yellow",  0),
        "predicted_total_yellow":   p.get("total_yellow", 0),
        "predicted_home_red":       p.get("home_red",     0),
        "predicted_away_red":       p.get("away_red",     0),
        "predicted_total_red":      p.get("total_red",    0),
        "predicted_home_corners":   p.get("home_corners", 0),
        "predicted_away_corners":   p.get("away_corners", 0),
        "predicted_total_corners":  p.get("total_corners",0),
    })

group_pred_df = pd.DataFrame(group_pred_rows)

# ─────────────────────────────────────────────────────────────────────────────
# 11. OUTPUT: GROUP STANDINGS CSV (from MC)
# ─────────────────────────────────────────────────────────────────────────────

standing_rows = []
for team in all_wc_teams:
    grp = team_group.get(team, "?")
    rc  = group_rank_counts[team]
    standing_rows.append({
        "group":        grp,
        "team":         team,
        "p_1st":        round(rc.get("1", 0) / N_SIMS, 4),
        "p_2nd":        round(rc.get("2", 0) / N_SIMS, 4),
        "p_3rd":        round(rc.get("3", 0) / N_SIMS, 4),
        "p_4th":        round(rc.get("4", 0) / N_SIMS, 4),
        "p_qualify":    round((rc.get("1", 0) + rc.get("2", 0)) / N_SIMS, 4),
    })

standing_df = (
    pd.DataFrame(standing_rows)
    .sort_values(["group", "p_1st"], ascending=[True, False])
)

In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# 12. OUTPUT: KNOCKOUT BRACKET CSV — deterministic bracket propagation
# ─────────────────────────────────────────────────────────────────────────────
#
# Strategy:
#   Step A — resolve R32 home/away teams from MC frequencies with conflict
#             detection (no team duplicated across slots).
#   Step B — propagate winners/losers sequentially through R16 -> QF -> SF ->
#             3rd-place playoff -> Final using symmetrised neutral lambdas.
#             This guarantees a team eliminated in QF cannot appear in the SF.

def _pick_team(counts_dict, already_assigned):
    for team, _ in sorted(counts_dict.items(), key=lambda x: -x[1]):
        if team not in already_assigned:
            return team
    return "TBD"

def _ko_predict(home, away):
    lh_ha, la_ha, lh_ah, la_ah = [], [], [], []
    for tag, (m_home, m_away, dc_params, t2i) in models.items():
        seed = hist_s1 if tag == "s1" else hist_p2
        fr_ha = build_fixture_frame(home, away, KNOCKOUT_DATE, seed, is_neutral=1)
        dc_h, dc_a = get_dc_lambdas_safe(home, away, dc_params, t2i)
        X_ha = build_features(fr_ha.reset_index(drop=True), np.array([dc_h]), np.array([dc_a]))
        lh_ha.append(float(np.clip(m_home.predict(X_ha), 0.05, 10)[0]))
        la_ha.append(float(np.clip(m_away.predict(X_ha), 0.05, 10)[0]))
        fr_ah = build_fixture_frame(away, home, KNOCKOUT_DATE, seed, is_neutral=1)
        dc_h2, dc_a2 = get_dc_lambdas_safe(away, home, dc_params, t2i)
        X_ah = build_features(fr_ah.reset_index(drop=True), np.array([dc_h2]), np.array([dc_a2]))
        lh_ah.append(float(np.clip(m_home.predict(X_ah), 0.05, 10)[0]))
        la_ah.append(float(np.clip(m_away.predict(X_ah), 0.05, 10)[0]))
    lh = (np.mean(lh_ha) + np.mean(la_ah)) / 2
    la = (np.mean(la_ha) + np.mean(lh_ah)) / 2
    pm  = poisson_prob_matrix(lh, la)
    prb = outcome_probs(pm)
    ph, pa = np.unravel_index(np.argmax(pm), pm.shape)
    winner = home if prb[0] >= prb[2] else away
    loser  = away if winner == home else home
    p_win  = prb[0] if winner == home else prb[2]
    return winner, loser, f"{ph}-{pa}", lh, la, round(float(p_win), 4)

# Step A: resolve R32 teams, highest-confidence first
_r32_mids = list(
    knockout_matches[knockout_matches["round"] == "Round of 32"]["match_id"].astype(int)
)
_r32_mids.sort(key=lambda m: -(
    max(ko_home_counts[m].values(), default=0) +
    max(ko_away_counts[m].values(), default=0)
))
_assigned = set()
_r32_home = {}
_r32_away = {}
for _mid in _r32_mids:
    _h = _pick_team(ko_home_counts[_mid], _assigned)
    if _h != "TBD": _assigned.add(_h)
    _a = _pick_team(ko_away_counts[_mid], _assigned)
    if _a != "TBD": _assigned.add(_a)
    _r32_home[_mid] = _h
    _r32_away[_mid] = _a

# Step B: seed slot->team map from knockout_slots schema
_slot_team = {}
for _mid in _r32_mids:
    _ks_row = knockout_matches[knockout_matches["match_id"] == _mid].iloc[0]
    _slot_team[_ks_row["slot_home"]] = _r32_home[_mid]
    _slot_team[_ks_row["slot_away"]] = _r32_away[_mid]

_ROUNDS = ["Round of 32", "Round of 16", "Quarter-final",
           "Semi-final", "Third-place playoff", "Final"]

_bracket = {}

for _rnd in _ROUNDS:
    for _, _ks in knockout_matches[knockout_matches["round"] == _rnd].sort_values("match_id").iterrows():
        _mid  = int(_ks["match_id"])
        _home = _slot_team.get(_ks["slot_home"], "TBD")
        _away = _slot_team.get(_ks["slot_away"], "TBD")
        if _home != "TBD" and _away != "TBD":
            _w, _l, _sc, _lh, _la, _pw = _ko_predict(_home, _away)
        else:
            _w = _l = "TBD"; _sc = "TBD"; _lh = 1.2; _la = 0.9; _pw = 0.5
        _bracket[_mid] = {"home": _home, "away": _away, "winner": _w, "loser": _l,
                          "score": _sc, "lh": _lh, "la": _la, "p_win": _pw}
        _slot_team[f"Winner Match {_mid}"] = _w
        _slot_team[f"Loser Match {_mid}"]  = _l

bracket_rows = []
for _, row in knockout_matches.sort_values("match_id").iterrows():
    mid  = int(row["match_id"])
    bm   = _bracket.get(mid, {})
    pred_home_team = bm.get("home", "TBD")
    pred_away_team = bm.get("away", "TBD")
    most_likely_winner = bm.get("winner", "TBD")
    det_score = bm.get("score", "TBD")
    win_pct   = bm.get("p_win", 0.0)

    # Penalty prediction from MC frequency
    p_pens = round(ko_pens_counts[mid] / N_SIMS, 4)
    pred_penalties = p_pens > 0.35   # threshold above historical ~28% mean; False dominates EV

    # Winner side
    if most_likely_winner == pred_home_team:
        pred_winner_side = "home"
    elif most_likely_winner == pred_away_team:
        pred_winner_side = "away"
    else:
        pred_winner_side = "home"

    # Cards & corners for this matchup
    if pred_home_team != "TBD" and pred_away_team != "TBD":
        ko_cc = predict_cc_match(pred_home_team, pred_away_team, _cc_team_stats, _cc_globals)
    else:
        ko_cc = {"total_yellow": 4, "total_red": 0, "total_corners": 9}

    bracket_rows.append({
        "match_id":              mid,
        "round":                 row["round"],
        "date_utc":              row["date_utc"],
        "venue":                 row["venue"],
        "slot_home":             row["slot_home"],
        "slot_away":             row["slot_away"],
        "predicted_home_team":   pred_home_team,
        "predicted_away_team":   pred_away_team,
        "predicted_winner":      most_likely_winner,
        "predicted_winner_side": pred_winner_side,
        "winner_pct":            win_pct,
        "predicted_score":       det_score,
        "p_penalties":           p_pens,
        "predicted_penalties":   pred_penalties,
        "predicted_total_yellow":  ko_cc.get("total_yellow",  4),
        "predicted_total_red":     ko_cc.get("total_red",     0),
        "predicted_total_corners": ko_cc.get("total_corners", 9),
    })

bracket_df = pd.DataFrame(bracket_rows)

prob_rows = []
for team in all_wc_teams:
    prob_rows.append({
        "team":            team,
        "group":           team_group.get(team, "?"),
        "p_qualify_group": round((group_rank_counts[team].get("1", 0) +
                                  group_rank_counts[team].get("2", 0)) / N_SIMS, 4),
        "p_r32":           round(advance_counts["Round of 32"][team] / N_SIMS, 4),
        "p_r16":           round(advance_counts["Round of 16"][team] / N_SIMS, 4),
        "p_qf":            round(advance_counts["Quarter-final"][team] / N_SIMS, 4),
        "p_sf":            round(advance_counts["Semi-final"][team] / N_SIMS, 4),
        "p_final":         round(advance_counts["Final"][team] / N_SIMS, 4),
        "p_win":           round(advance_counts["Winner"][team] / N_SIMS, 4),
    })

prob_df = (
    pd.DataFrame(prob_rows)
    .sort_values("p_win", ascending=False)
    .reset_index(drop=True)
)

In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# 14. FULL COMPETITION SUBMISSION (104 matches)
# ─────────────────────────────────────────────────────────────────────────────

submission_rows = []

# ── Group stage (matches 1–72) ───────────────────────────────────────────────
for mid in match_ids_ordered:
    p = group_preds[mid]

    _h, _a = map(int, p["predicted_score"].split("-"))
    if _h > _a:
        pred_winner = "home"
    elif _a > _h:
        pred_winner = "away"
    else:
        pred_winner = "draw"

    submission_rows.append({
        "match_id":           mid,
        "round":              "Group Stage",
        "multiplier":         1,
        "home_team":          p["home_team"],
        "away_team":          p["away_team"],
        "score":              p["predicted_score"],
        "total_corners":      p.get("total_corners", 9),
        "total_yellow_cards": p.get("total_yellow",  4),
        "total_red_cards":    p.get("total_red",     0),
        "winning_team":       pred_winner,
        "matchup_home":       "",
        "matchup_away":       "",
        "match_winner":       "",
        "penalties":          "",
    })

# ── Knockout stage (matches 73–104) ─────────────────────────────────────────
for _, brow in bracket_df.iterrows():
    mid       = int(brow["match_id"])
    pred_home = brow["predicted_home_team"]
    pred_away = brow["predicted_away_team"]
    winner    = brow["predicted_winner"]

    if winner == pred_home:
        winner_side = "home"
    elif winner == pred_away:
        winner_side = "away"
    else:
        winner_side = "home"

    mult_row = knockout_matches.loc[knockout_matches["match_id"] == mid, "multiplier"]
    mult = int(mult_row.iloc[0]) if not mult_row.empty else 1

    submission_rows.append({
        "match_id":           mid,
        "round":              brow["round"],
        "multiplier":         mult,
        "home_team":          pred_home,
        "away_team":          pred_away,
        "score":              brow["predicted_score"],
        "total_corners":      int(brow["predicted_total_corners"]),
        "total_yellow_cards": int(brow["predicted_total_yellow"]),
        "total_red_cards":    int(brow["predicted_total_red"]),
        "winning_team":       "",
        "matchup_home":       pred_home,
        "matchup_away":       pred_away,
        "match_winner":       winner_side,
        "penalties":          str(brow["predicted_penalties"]),
    })

submission_df = (
    pd.DataFrame(submission_rows)
    .sort_values("match_id")
    .reset_index(drop=True)
)

## 🗓️ Group stage predictions

Fill in your predictions for all 72 group stage matches below.

In [27]:
group_predictions = group_fixtures.copy()

# Fill in your predictions for each match
# Example (match 1 — Mexico vs South Africa): predicted_home_goals=2, predicted_away_goals=1, corners=9, yellow_cards=3, red_cards=0, winning_team='home'
group_predictions['predicted_home_goals'] = None   # e.g. 2
group_predictions['predicted_away_goals'] = None   # e.g. 1
group_predictions['corners']              = None   # e.g. 9
group_predictions['yellow_cards']         = None   # e.g. 3
group_predictions['red_cards']            = None   # e.g. 0
group_predictions['winning_team']         = None   # "home", "away", or "draw"

# ── Fill group stage template ────────────────────────────────────────────────
grp_sub = submission_df[submission_df["round"] == "Group Stage"].copy()
grp_sub[["score_h", "score_a"]] = grp_sub["score"].str.split("-", expand=True).astype(int)
grp_sub = grp_sub.reset_index(drop=True)

group_predictions["predicted_home_goals"] = grp_sub["score_h"]
group_predictions["predicted_away_goals"] = grp_sub["score_a"]
group_predictions["corners"]              = grp_sub["total_corners"]
group_predictions["yellow_cards"]         = grp_sub["total_yellow_cards"]
group_predictions["red_cards"]            = grp_sub["total_red_cards"]
group_predictions['winning_team']         = grp_sub["winning_team"]
group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",1,0,9,4,0,home
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",1,0,13,5,0,home
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",2,0,12,6,0,home
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",1,0,9,4,0,home
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",1,0,11,3,0,home
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",1,0,8,5,0,home
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",1,0,9,3,0,home
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",0,1,9,3,0,away
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",1,1,9,5,0,draw


## 🏆 Knockout stage predictions

For knockout matches you also predict **which teams are playing**. Fill in the team names based on your group stage predictions, then add your match predictions.

In [28]:
knockout_predictions = knockout_slots.copy()

# Fill in your predictions for each knockout match
# Example (match 73 — Round of 32): predicted_home_team='Brazil', predicted_away_team='France', predicted_home_goals=1, predicted_away_goals=0, corners=8, yellow_cards=2, red_cards=0, match_winner='home', penalties=False
knockout_predictions['predicted_home_team']  = None   # e.g. "Brazil"
knockout_predictions['predicted_away_team']  = None   # e.g. "France"
knockout_predictions['predicted_home_goals'] = None   # e.g. 1
knockout_predictions['predicted_away_goals'] = None   # e.g. 0
knockout_predictions['corners']              = None   # e.g. 8
knockout_predictions['yellow_cards']         = None   # e.g. 2
knockout_predictions['red_cards']            = None   # e.g. 0
knockout_predictions['match_winner']         = None   # "home" or "away"
knockout_predictions['penalties']            = None   # True or False

# ── Fill knockout template ───────────────────────────────────────────────────
ko_sub = submission_df[submission_df["round"] != "Group Stage"]
ko_sub[["score_h", "score_a"]] = ko_sub["score"].str.split("-", expand=True).astype(int)
ko_sub = ko_sub.reset_index(drop=True)

knockout_predictions["predicted_home_team"]  = ko_sub["home_team"]
knockout_predictions["predicted_away_team"]  = ko_sub["away_team"]
knockout_predictions["predicted_home_goals"] = ko_sub["score_h"]
knockout_predictions["predicted_away_goals"] = ko_sub["score_a"]
knockout_predictions["corners"]              = ko_sub["total_corners"]
knockout_predictions["yellow_cards"]         = ko_sub["total_yellow_cards"]
knockout_predictions["red_cards"]            = ko_sub["total_red_cards"]
knockout_predictions['match_winner']         = ko_sub["match_winner"]
knockout_predictions['penalties']            = ko_sub["penalties"]
knockout_predictions

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,Mexico,Canada,1,0,9,4,0,away,False
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,Brazil,Japan,1,1,13,3,0,home,False
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Germany,Czech Republic,1,0,12,3,0,home,False
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,Netherlands,Morocco,1,1,7,3,0,home,False
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,Ecuador,Senegal,1,0,10,3,1,home,False
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),France,Qatar,1,0,10,3,0,home,False
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),South Korea,Ivory Coast,1,0,12,5,0,home,False
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),England,Ivory Coast,1,0,9,3,0,home,False
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),Belgium,Saudi Arabia,1,1,8,4,0,home,False
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),United States,Saudi Arabia,1,0,8,4,0,home,False


In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# 15. PREDICTION SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  TOURNAMENT WINNER PREDICTION  (top 10 by P(Win))")
print("=" * 70)
print(f"  {'Team':<28} {'Group':<8} {'P(QF)':<9} {'P(SF)':<9} {'P(Final)':<10} {'P(Win)'}")
print("-" * 70)
for _, row in prob_df.head(10).iterrows():
    print(f"  {row['team']:<28} {row['group']:<8} "
          f"{row['p_qf']:<9.1%} {row['p_sf']:<9.1%} "
          f"{row['p_final']:<10.1%} {row['p_win']:.1%}")

print("\n  Predicted winner:", prob_df.iloc[0]["team"],
      f"(P={prob_df.iloc[0]['p_win']:.1%})")

# Identify most likely finalists from SF winners
sf_winner_counts = defaultdict(int)
for mid in sf_mids:
    for team, cnt in ko_bracket_wins[mid].items():
        sf_winner_counts[team] += cnt
top_finalists = sorted(sf_winner_counts, key=lambda t: -sf_winner_counts[t])[:2]
finalist1 = top_finalists[0] if len(top_finalists) > 0 else "TBD"
finalist2 = top_finalists[1] if len(top_finalists) > 1 else "TBD"
final_winner = prob_df.iloc[0]["team"]
final_score_row = bracket_df[bracket_df["match_id"] == 104]
final_score = final_score_row.iloc[0]["predicted_score"] if not final_score_row.empty else "?-?"
print(f"  Predicted final : {finalist1} vs {finalist2} -> {final_winner} {final_score}")


  TOURNAMENT WINNER PREDICTION  (top 10 by P(Win))
  Team                         Group    P(QF)     P(SF)     P(Final)   P(Win)
----------------------------------------------------------------------
  Spain                        H        19.6%     10.9%     6.1%       6.1%
  Brazil                       C        18.7%     10.5%     6.0%       6.0%
  Argentina                    J        16.4%     9.9%      5.8%       5.8%
  Uruguay                      H        18.4%     10.2%     5.4%       5.4%
  France                       I        17.8%     9.1%      4.9%       4.9%
  Colombia                     K        13.8%     7.9%      4.2%       4.2%
  United States                D        14.1%     7.8%      4.0%       4.0%
  Iran                         G        12.3%     7.0%      3.7%       3.7%
  Canada                       B        12.6%     6.8%      3.5%       3.5%
  England                      L        13.5%     6.6%      3.3%       3.3%

  Predicted winner: Spain (P=6.1%)
  P

## ✅ Checklist before publishing into the competition

- Rename your workspace to make it descriptive of your work. N.B. you should leave the notebook name as `notebook.ipynb`.
- Remove redundant cells like the judging criteria, so the workbook is focused on your predictions.
- Make sure all prediction cells are filled in—`None` values will score 0 points.
- Check that all cells run without error.
- Make sure your workbook is published before **June 10, 2026 at 09:00 UTC**.

## ⏳ Time is ticking. Good luck!